# Evaluation script for the new GPT planners

In [1]:
import sys
import os
import json
import yaml
import pathlib
import glob
import pandas as pd
from pathlib import Path

In [2]:
# Replace 'experiment_log.json' with the path to your actual JSON file
LOCAL_MACHINE = "am1"
ROOT_PATH = pathlib.Path("__file__").resolve().parent.parent
EXP_FOLDER = os.path.join(ROOT_PATH, "experiments")
CONFIGS_FOLDER = os.path.join(ROOT_PATH, "cos_eor", "configs", "local")
ENVS_FILE_PATH = os.path.join(CONFIGS_FOLDER , "envs_paper.yaml")

# Note: put the OpenAI key here:
with open(os.path.join(CONFIGS_FOLDER, "api_key.yaml")) as kfile:
    k = yaml.safe_load(kfile)

In [3]:
# ==========================================
# 1. 기본 설정 (Base Settings)
# ==========================================
expand_config = {
    'pomaria_1_int': {'base_name': 'Scene 1', 'suffixes': [2, 3, 4]},
    'merom_1_int': {'base_name': 'Scene 2', 'suffixes': [2, 3, 4]},
    'pomaria_2_int': {'base_name': 'Scene 3', 'suffixes': [2, 3, 4]},
    'rs_int': {'base_name': 'Scene 4', 'suffixes': [2, 3, 4]}  
}

PAIRS = {
    'merom_1_int': {'bootstrap': 'IL_scene2_4.1_v7_cp8_pair_eval', 'finetune iter 1': 'IT1_scene2_4.1_v3_cp1_pair_eval_v2', 'finetune iter 2': 'IT2_scene2_4.1_v1_cp2_pair_eval_v2', 'LITEPlanner': 'ours_scene2_pair_eval'},
    'pomaria_1_int': {'bootstrap': 'IL_scene1_4.1_v7_cp8_pair_eval', 'finetune iter 1': 'IT1_scene1_4.1_v1_cp2_pair_eval', 'finetune iter 2': 'IT2_scene1_4.1_v1_cp2_pair_eval', 'LITEPlanner': 'ours_scene1_pair_eval_v3'},
    'pomaria_2_int': {'bootstrap': 'IL_scene3_4.1_v8_4e_cp2_pair_eval', 'finetune iter 1':'IT1_scene3_4.1_v1_cp1_pair_eval', 'finetune iter 2':'IT2_scene3_4.1_v1_cp2_pair_eval', 'LITEPlanner': 'ours_scene3_pair_eval'},
    'rs_int': {'bootstrap': 'IL_scene4_4.1_v8_4e_cp2_pair_eval', 'finetune iter 1': 'IT1_scene4_4.1_v1_cp2_pair_eval', 'finetune iter 2':'IT2_scene4_4.1_v1_cp2_pair_eval', 'LITEPlanner': 'ours_scene4_pair_eval'},
}

name_map = {
    'pomaria_1_int': 'Scene 1',
    'merom_1_int': 'Scene 2',
    'pomaria_2_int': 'Scene 3',
    'rs_int': 'Scene 4'
}

SINGLES = ['single_run_20_turbo_benevolence1_pomaria1_1', 'single_run_22_turbo_other_5_scenes_1']
SAYPLAN = ['sayplan_paper_1']
SAYCAN = ['saycan_paper_1']

NUM_TRAIN_SCENES = 8
NUM_TEST_SCENES = 0

# yaml 읽기
with open(ENVS_FILE_PATH , 'r') as file:
    scenes_list = yaml.safe_load(file).split()
scenes = {}
for i, s in enumerate(scenes_list):
    if i < NUM_TRAIN_SCENES:
        scenes[s] = "train"
    else:
        scenes[s] = "test"


# ==========================================
# 2. 자동화 루프 (x2, x3 등 폴더를 '먼저' 추가합니다!)
# ==========================================
scene_order = []

for base_scene, config in expand_config.items():
    base_name = config['base_name']
    scene_order.append(base_name)
    
    for x in config['suffixes']:
        new_scene_id = f"{base_scene}_x{x}"
        new_display_name = f"{base_name} x{x}"
        
        folder_idx = base_name.replace('Scene ', '')
        base_it2_folder = PAIRS[base_scene].get('finetune iter 2', '')
        
        PAIRS[new_scene_id] = {
            'LITEPlanner': f'ours_scene{folder_idx}_pair_eval_x{x}_v2'
        }
        
        if base_it2_folder:
            PAIRS[new_scene_id]['finetune iter 2'] = f"{base_it2_folder}_x{x}_v2"
        
        scenes[new_scene_id] = "train"
        name_map[new_scene_id] = new_display_name
        scene_order.append(new_display_name)

# 확장 안 된 나머지 씬들도 순서에 추가
for base_scene, display_name in name_map.items():
    if display_name not in scene_order:
        scene_order.append(display_name)

print('scenes updated:', scenes)


# ==========================================
# 3. 매핑 및 최종 로드 목록 생성 (모든 폴더가 추가된 후 실행됨!)
# ==========================================
VARIANT_NAME_MAPPING = {}
SOURCE_MAPPING = {}
for s in PAIRS:
    for name in PAIRS[s]:
        variant = PAIRS[s][name]
        VARIANT_NAME_MAPPING[variant] = name + ' | ' + s
        SOURCE_MAPPING[variant] = s
        
for v in SINGLES:
    SOURCE_MAPPING[v] = 'None'
    VARIANT_NAME_MAPPING[v] = 'zero-shot gpt-3.5-turbo'
for v in SAYPLAN:
    SOURCE_MAPPING[v] = 'None'
    VARIANT_NAME_MAPPING[v] = 'SayPlan'
for v in SAYCAN:
    SOURCE_MAPPING[v] = 'None'
    VARIANT_NAME_MAPPING[v] = 'SayCan'

def flatten_list(input_list):
    return [item for sublist in input_list for item in sublist]

SOURCE_SCENES = ['merom_1_int', 'pomaria_1_int', 'pomaria_2_int', 'rs_int']

# ★ 이제 여기에 x2, x3 등 추가된 폴더명들이 모두 정상적으로 들어갑니다! ★
COMPARE_EXP = list(VARIANT_NAME_MAPPING.keys()) 

LOGS_FOLDER = os.path.join(ROOT_PATH, "logs")

# Constants
ANNOTATION = "annotation"
DIFF_CORRECT_LOC = "diff_correct_loc"
EXPERIMENT = "experiment"
FLAG = "flag"
NUM_OBJECTS_DISCOVERED = "num_objects_discovered"
NUM_RECS_DISCOVERED = "num_recs_discovered"
OUTCOME = "outcome"
PROMPT = "prompt"
REWARD = "reward"
REWARD_WEIGHTS = {NUM_OBJECTS_DISCOVERED: 1, NUM_RECS_DISCOVERED: 1, DIFF_CORRECT_LOC: 10}
SCENE = "scene"
SPLIT = "split"
SPLIT_SCENE = "split_scene"
SUC = "succeeded"
SUC_STEPS = "successful_steps"

scenes updated: {'pomaria_1_int': 'train', 'pomaria_2_int': 'train', 'merom_1_int': 'train', 'rs_int': 'train', 'pomaria_1_int_x2': 'train', 'pomaria_1_int_x3': 'train', 'pomaria_1_int_x4': 'train', 'merom_1_int_x2': 'train', 'merom_1_int_x3': 'train', 'merom_1_int_x4': 'train', 'pomaria_2_int_x2': 'train', 'pomaria_2_int_x3': 'train', 'pomaria_2_int_x4': 'train', 'rs_int_x2': 'train', 'rs_int_x3': 'train', 'rs_int_x4': 'train'}


In [4]:
# Load the scene IDs from the envs.yaml file
def load_scenes(file_path):
    with open(file_path, 'r') as file:
        return yaml.safe_load(file)

# Function to get log file paths for a given experiment ID and scene ID
def get_experiment_log_paths(experiment_id, scene_id):
    folder_name = scene_id.split('_x')[0] if '_x' in scene_id else scene_id
    
    logs_path_pattern = f'{LOGS_FOLDER}/{experiment_id}/demo/{folder_name}/data_*.json'    
    # logs_path_pattern = f'{LOGS_FOLDER}/{experiment_id}/demo/{scene_id}/data_*.json'
    MOD_VAL = 10
    if 'ablation' in experiment_id:
        MOD_VAL = 35
        MOD_TEST = 10
    elif 'sayplan' in experiment_id or 'saycan' in experiment_id:
        MOD_VAL = 25
        MOD_TEST = 10
    elif 'pair' in experiment_id:
        MOD_VAL = 25
        MOD_TEST = 10
    elif 'small' in experiment_id:
        MOD_VAL = 10
        MOD_TEST = 10
    else:
        MOD_VAL = 10
        MOD_TEST = 10
    
    all_paths = sorted(glob.glob(logs_path_pattern), reverse=False)[:] # sorted ascending according to time
    
    log_paths = {}
    if scenes[scene_id] == 'test':
        for i, path in enumerate(all_paths):
            if i % MOD_TEST <= 9:
                log_paths[path] = 'test'
    elif scenes[scene_id] == 'train':
        # half of the paths are train/test iterations
        for i, path in enumerate(all_paths):
            if i % MOD_VAL >= 5:
                # log 5 - 9,14,24 are for training
                log_paths[path] = 'train'
            else:
                # log 0 - 4 are for testing
                log_paths[path] = 'test'
    return log_paths


def reward(result):
    reward = sum(result[k] * REWARD_WEIGHTS[k] for k in REWARD_WEIGHTS)
    return reward

def annotate_record(record, experiment_id, scene_id):
    result = {}
    result[SCENE] = scene_id
    result[EXPERIMENT] = experiment_id
    result[NUM_OBJECTS_DISCOVERED] = len(record[OUTCOME]["objects_discovered"])
    result[NUM_RECS_DISCOVERED] = len(record[OUTCOME]["recs_discovered"])
    result[DIFF_CORRECT_LOC] = record[OUTCOME]["count_correct"]["end"] - record[OUTCOME]["count_correct"]["start"]
    # craft a response based on successful steps
    result[REWARD] = reward(result)
    return result

def annotate_episode(records, experiment_id, scene_id, split='train'):
    lite_stats = {}
    last_record = records[-1]
        
    if "total_calls" in last_record:
        lite_stats = last_record["total_calls"]
        records = records[:-1] 
    else:
        pass
    
    if len(records) == 0:
        # 데이터가 없는 껍데기 파일인 경우 모두 0으로 처리해서 반환
        return {
            'success_rate': 0, 'objects_discovered': 0, 'recs_discovered': 0,
            'experiment': experiment_id, 'scene': scene_id, 'split': split,
            'llm_calls': lite_stats.get('total_llm_calls', 0),
            'ppr_calls': lite_stats.get('total_ppr_calls', 0),
            'total_steps': 0, 'steps': 0, 'successful_steps': 0,
            'pick_steps': 0, 'correct_placement_steps': 0, 'wrong_placement_steps': 0,
            'placement_steps': 0, 'movements': [], 'diff_gt': 0, 'diff_correct_loc': 0
        }
        
    total_steps = records[-1].get("end", 0)
    
    result = {}
    start_correct = records[0][OUTCOME]["count_correct"]["start"]
    start_wrong = records[0][OUTCOME]["count_wrong"]["start"]
    end_correct = records[-1][OUTCOME]["count_correct"]["end"]
        
    diff_correct = end_correct - start_correct
    objs = []
    recs = []
    for record in records:
        objs += record[OUTCOME]["objects_discovered"]
        recs += record[OUTCOME]["recs_discovered"]
    result[DIFF_CORRECT_LOC] = diff_correct
    result['diff_gt'] = start_wrong
    result['success_rate'] = diff_correct/start_wrong * 100
    result["objects_discovered"] = len(objs)
    result['recs_discovered'] = len(recs)
    result['experiment'] = experiment_id
    result['scene'] = scene_id
    
    low_level_count = 0
    for record in records:
        if isinstance(record, dict):
            if record.get("low_level") and record["low_level"].get("response"):
                low_level_count += 1
    
    personalize_exp_ids = [val for s_dict in PAIRS.values() for val in s_dict.values()]
    
    if experiment_id in personalize_exp_ids:
        result['llm_calls'] = low_level_count
    else:
        result['llm_calls'] = lite_stats.get('total_llm_calls', 0)
    
    result['ppr_calls'] = lite_stats.get('total_ppr_calls', 0)
    
    result['total_steps'] = total_steps
    
    steps = 0
    successful_steps = 0
    for record in records:
        steps += len(record["logs"])
        successful_steps += len([l for l in record["logs"] if l['flag'] == SUC])
        
    num_correct_placement = 0
    num_wrong_placement = 0
    num_placement = 0
    num_pick = 0
    movements = []
    for record in records:
        for log in record['logs']:
            diff = log[OUTCOME]["count_correct"]['end'] - log[OUTCOME]["count_correct"]['start']
            objects_moved = log[OUTCOME]["objects_moved"]
            for obj in objects_moved:
                if objects_moved[obj][-1] == "agent":
                    # picking only, could be right or wrong
                    num_pick += 1
                else: 
                    # moving from agent to rec or moving from rec to rec
                    num_placement += 1
                    if objects_moved[obj][0] == 'agent':
                        # moving from agent to rec
                        if diff <= 0:
                            # wrong placement
                            num_wrong_placement += 1
                        else:
                            num_correct_placement += 1

                    else:
                        # moving from rec to rec
                        if diff < 0:
                            num_wrong_placement += 1
                        else:
                            # +1 if wrong rec -> correct rec, 0 if correct rec -> correct rec
                            num_correct_placement += 1
                            
                    movements.append((obj, objects_moved[obj][-1]))
                    
            
    result['steps'] = steps
    result['successful_steps'] = successful_steps
    result['pick_steps'] = num_pick
    result['correct_placement_steps'] = num_correct_placement
    result['wrong_placement_steps'] = num_wrong_placement
    result['placement_steps'] = num_placement
    result['split'] = split
    result['movements'] = movements
    return result
    

# Function to load experiment logs and add the scene name
def load_and_annotate_logs(experiment_id, scenes):
    all_records = []
    all_episodes = []
    for scene_id in scenes:
        # log file paths is a dictionary {path: 'train' or 'test'}
        log_file_paths = get_experiment_log_paths(experiment_id, scene_id)
        for log_file_path in log_file_paths:
            with open(log_file_path, 'r') as file:
                records = json.load(file)[:]
                split = log_file_paths[log_file_path]
                episode_result = annotate_episode(records, experiment_id, scene_id, split)
                all_episodes.append(episode_result)
                
                if records and "total_calls" in records[-1]:
                    records = records[:-1]
                    
                # Annotate each record with the scene name
                for record in records:
                    record[ANNOTATION] = annotate_record(record, experiment_id, scene_id)
                all_records.extend(records)
    return all_records, all_episodes

# Load and annotate logs
annotated_logs = {}
annotated_episodes = {}
annotated_episode_list = []
annotations = []
for experiment_id in COMPARE_EXP:
    annotated_logs[experiment_id], annotated_episodes[experiment_id] = load_and_annotate_logs(experiment_id, scenes)
    annotated_episode_list += annotated_episodes[experiment_id]
    annotations += [l[ANNOTATION] for l in annotated_logs[experiment_id]]


In [5]:
import json

print("🔍 빈 로그 파일 탐색을 시작합니다...\n")
empty_files = []

# COMPARE_EXP에 등록된 모든 실험과 씬을 순회합니다.
for experiment_id in COMPARE_EXP:
    for scene_id in scenes:
        # 해당 실험/씬의 모든 로그 파일 경로를 가져옵니다.
        log_file_paths = get_experiment_log_paths(experiment_id, scene_id)
        
        for log_file_path in log_file_paths:
            try:
                with open(log_file_path, 'r') as file:
                    records = json.load(file)
                    
                    # 1. 파일이 아예 비어있는 경우 ([])
                    # 2. 데이터가 1개뿐인데 그게 "total_calls" 블록인 경우
                    if len(records) == 0 or (len(records) == 1 and "total_calls" in records[0]):
                        empty_files.append(log_file_path)
                        print(f"⚠️ 빈 데이터 발견: {log_file_path}")
                        
            except json.JSONDecodeError:
                print(f"❌ JSON 파싱 에러 (파일이 손상됨): {log_file_path}")
            except Exception as e:
                print(f"❌ 파일을 읽는 중 에러 발생 ({log_file_path}): {e}")

print(f"\n✅ 탐색 완료! 총 {len(empty_files)}개의 빈 로그 파일을 찾았습니다.")

🔍 빈 로그 파일 탐색을 시작합니다...


✅ 탐색 완료! 총 0개의 빈 로그 파일을 찾았습니다.


In [6]:
df_episodes = pd.DataFrame(annotated_episode_list)

df_episodes['total_calls'] = df_episodes.apply(lambda x: f"{x['llm_calls']} / {x['ppr_calls']}", axis=1)

df_episodes[SPLIT_SCENE] = df_episodes.apply(lambda x: scenes[x[SCENE]], axis=1)
df_episodes['variant'] = df_episodes.apply(lambda x: VARIANT_NAME_MAPPING[x['experiment']], axis=1)
df_episodes['source'] = df_episodes.apply(lambda x: SOURCE_MAPPING[x[EXPERIMENT]], axis=1)

In [7]:
pd.set_option('display.max_rows', 200)
train_instances = df_episodes[df_episodes.apply(lambda x: x['source'] == 'None' or x['source'] == x['scene'] , axis=1)]

# Results for the paper

### The main results: in-domain adaptation success metric

In [8]:
df_main = train_instances.copy(deep=True)

In [9]:
# scene 1: pomaria_1
# scene 2: merom_1
# scene 3: rs_int
# scene 4: pomaria_2

variant_map = {
    'zero-shot gpt-3.5-turbo': 'LLM-Planner',
    'SayPlan': 'SayPlan',
    'SayPlan-nofeed': 'SayPlan-nofeed',
    'SayCan': 'SayCan',
    'bootstrap': 'LLM-Personalize (Imitation learning)',
    'finetune iter 1': 'LLM-Personalize (SI Iter=1)',
    'finetune iter 2': 'LLM-Personalize (SI Iter=2)',
    'LITEPlanner': 'LITE-Planner',
    'bt10': 'ablation_bt_large_1',
    'bt11': 'ablation_bt_small_1',
    'bt12': 'ablation_bt_large_2',
    'bt14': 'ablation_bt_large_3',
    'ft31': 'ablation_ft_small_iter1'
}
df_main['scene name'] = df_main.apply(lambda x: name_map[x['scene']], axis=1)
df_main['variant name'] = df_main.apply(lambda x: variant_map[x['variant'].split('|')[0].strip(' ')], axis=1)
df_main['task set'] = df_main.apply(lambda x: x['split'], axis=1)
df_main['scene name'] = pd.Categorical(df_main['scene name'], categories=scene_order, ordered=True)
# Assuming df_main is your original DataFrame
# Step 1: Group by and aggregate
grouped = df_main.groupby(['scene name', 'task set', 'variant name'])['success_rate'].agg(['mean', 'sem', 'count'])
# Step 2: Unstack to rearrange the DataFrame
# Unstack 'scene name' and 'task set' to create a multi-level column structure
reshaped = grouped.unstack(level=[0, 1])
# Step 3: Reorder the columns to ensure 'train' appears before 'test'
# This step might require custom handling based on the specific column names in your DataFrame
reshaped = reshaped.swaplevel(1, 2, axis=1).sort_index(axis=1)
reshaped = reshaped.swaplevel(0, 2, axis=1).sort_index(axis=1)
reshaped


scene name                           Scene 1                                   \
task set                                test                 train              
                                       count  mean       sem count       mean   
variant name                                                                    
LITE-Planner                              25  41.8  3.080584   100  26.966667   
LLM-Personalize (Imitation learning)      25  12.0  7.071068   100   5.350000   
LLM-Personalize (SI Iter=1)               25  26.2  8.308028   100  12.850000   
LLM-Personalize (SI Iter=2)               25  35.4  7.308215   100  27.250000   

scene name                                     Scene 1 x2                  \
task set                                             test                   
                                           sem      count  mean       sem   
variant name                                                                
LITE-Planner                          3.193127         25  41.8  2.785827   
LLM-Personalize (Imitation learning)  2.466759          0   NaN       NaN   
LLM-Personalize (SI Iter=1)           3.184497          0   NaN       NaN   
LLM-Personalize (SI Iter=2)           4.229617         25  41.5  3.122499   

scene name                                  ... Scene 4 x3                 \
task set                             train  ...       test train            
                                     count  ...        sem count mean sem   
variant name                                ...                             
LITE-Planner                           100  ...        NaN     0  NaN NaN   
LLM-Personalize (Imitation learning)     0  ...        NaN     0  NaN NaN   
LLM-Personalize (SI Iter=1)              0  ...        NaN     0  NaN NaN   
LLM-Personalize (SI Iter=2)            100  ...        NaN     0  NaN NaN   

scene name                           Scene 4 x4                          
task set                                   test          train           
                                          count mean sem count mean sem  
variant name                                                             
LITE-Planner                                  0  NaN NaN     0  NaN NaN  
LLM-Personalize (Imitation learning)          0  NaN NaN     0  NaN NaN  
LLM-Personalize (SI Iter=1)                   0  NaN NaN     0  NaN NaN  
LLM-Personalize (SI Iter=2)                   0  NaN NaN     0  NaN NaN  

[4 rows x 96 columns]

In [10]:
grouped1 = df_main.groupby(['scene name', 'task set', 'variant name'])['success_rate'].agg(
    lambda x: f'{x.mean():.2f} ± {x.sem():.2f}'
)
reshaped1 = grouped1.unstack(level=[0, 1])
desired_order = [
    'LLM-Personalize (Imitation learning)',
    'LLM-Personalize (SI Iter=1)', 
    'LLM-Personalize (SI Iter=2)', 
    'LITE-Planner'
]

reshaped1 = reshaped1.sort_index(axis=1, level=[0, 1], ascending=[True, False])
reshaped1 = reshaped1.reindex(desired_order)
reshaped1

scene name                                 Scene 1                \
task set                                     train          test   
variant name                                                       
LLM-Personalize (Imitation learning)   5.35 ± 2.47  12.00 ± 7.07   
LLM-Personalize (SI Iter=1)           12.85 ± 3.18  26.20 ± 8.31   
LLM-Personalize (SI Iter=2)           27.25 ± 4.23  35.40 ± 7.31   
LITE-Planner                          26.97 ± 3.19  41.80 ± 3.08   

scene name                              Scene 1 x2               Scene 1 x3  \
task set                                     train          test      train   
variant name                                                                  
LLM-Personalize (Imitation learning)           NaN           NaN        NaN   
LLM-Personalize (SI Iter=1)                    NaN           NaN        NaN   
LLM-Personalize (SI Iter=2)           40.21 ± 2.40  41.50 ± 3.12        NaN   
LITE-Planner                          25.90 ± 1.96  41.80 ± 2.79        NaN   

scene name                                Scene 1 x4            Scene 2  \
task set                             test      train test         train   
variant name                                                              
LLM-Personalize (Imitation learning)  NaN        NaN  NaN   5.13 ± 3.06   
LLM-Personalize (SI Iter=1)           NaN        NaN  NaN  13.98 ± 3.04   
LLM-Personalize (SI Iter=2)           NaN        NaN  NaN  21.13 ± 3.26   
LITE-Planner                          NaN        NaN  NaN  22.88 ± 2.30   

scene name                                          ... Scene 3 x4       \
task set                                      test  ...      train test   
variant name                                        ...                   
LLM-Personalize (Imitation learning)  16.80 ± 5.81  ...        NaN  NaN   
LLM-Personalize (SI Iter=1)           17.20 ± 5.34  ...        NaN  NaN   
LLM-Personalize (SI Iter=2)           20.67 ± 7.15  ...        NaN  NaN   
LITE-Planner                          20.80 ± 4.36  ...        NaN  NaN   

scene name                                 Scene 4               Scene 4 x2  \
task set                                     train          test      train   
variant name                                                                  
LLM-Personalize (Imitation learning)  28.67 ± 2.79  29.60 ± 5.72        NaN   
LLM-Personalize (SI Iter=1)           28.45 ± 3.40  39.13 ± 7.63        NaN   
LLM-Personalize (SI Iter=2)           42.07 ± 2.88  34.13 ± 6.99        NaN   
LITE-Planner                          43.32 ± 3.40  37.20 ± 4.54        NaN   

scene name                                Scene 4 x3      Scene 4 x4       
task set                             test      train test      train test  
variant name                                                               
LLM-Personalize (Imitation learning)  NaN        NaN  NaN        NaN  NaN  
LLM-Personalize (SI Iter=1)           NaN        NaN  NaN        NaN  NaN  
LLM-Personalize (SI Iter=2)           NaN        NaN  NaN        NaN  NaN  
LITE-Planner                          NaN        NaN  NaN        NaN  NaN  

[4 rows x 32 columns]

In [11]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# 1. 오리지널 씬과 확장 씬(x2, x3, x4) 쪼개기
original_scenes = ['Scene 1', 'Scene 2', 'Scene 3', 'Scene 4']

df_orig = df_main[df_main['scene name'].isin(original_scenes)].copy()
df_ext = df_main[~df_main['scene name'].isin(original_scenes)].copy()

# ★ 추가된 부분: 안 쓰는 카테고리(빈 컬럼)의 흔적을 아예 지워버립니다.
if hasattr(df_orig['scene name'], 'cat'):
    df_orig['scene name'] = df_orig['scene name'].cat.remove_unused_categories()
if hasattr(df_ext['scene name'], 'cat'):
    df_ext['scene name'] = df_ext['scene name'].cat.remove_unused_categories()

# ==========================================
# 함수: 성공률(Success Rate) 표 생성 함수
# ==========================================
def create_success_rate_table(df_subset):
    if df_subset.empty:
        return "데이터가 없습니다."
        
    def format_sr(x):
        if x.count() == 0:
            return np.nan # 나중에 빈 열을 한 번에 날리기 위해 임시로 NaN 처리
        elif x.count() == 1:
            return f"{x.mean():.2f} ± 0.00"
        else:
            return f"{x.mean():.2f} ± {x.sem():.2f}"

    grouped = df_subset.groupby(['scene name', 'task set', 'variant name'])['success_rate'].agg(format_sr)
    reshaped = grouped.unstack(level=[0, 1])
    
    # ★ 추가된 부분: 컬럼 전체가 NaN(데이터 없음)인 경우 열 자체를 삭제합니다.
    reshaped = reshaped.dropna(axis=1, how='all')
    
    # 남은 NaN값(일부 모델만 안 돌린 경우 등)은 보기 좋게 '-'로 변경
    reshaped = reshaped.fillna("-")
    
    # 컬럼 정렬 (Train 우선)
    reshaped = reshaped.sort_index(axis=1, level=[0, 1], ascending=[True, False])
    
    # 모델 순서 적용
    if 'desired_order' in globals():
        existing_rows = [r for r in desired_order if r in reshaped.index]
        reshaped = reshaped.reindex(existing_rows)
        
    return reshaped

# ==========================================
# 결과 출력
# ==========================================
print("=== [표 1] 오리지널 씬 성공률 (Scene 1 ~ 4) ===")
sr_table_orig = create_success_rate_table(df_orig)
display(sr_table_orig)

print("\n\n=== [표 2] 확장 씬 성공률 (x2, x3, x4) ===")
sr_table_ext = create_success_rate_table(df_ext)
display(sr_table_ext)

=== [표 1] 오리지널 씬 성공률 (Scene 1 ~ 4) ===


scene name                                 Scene 1                \
task set                                     train          test   
variant name                                                       
LLM-Personalize (Imitation learning)   5.35 ± 2.47  12.00 ± 7.07   
LLM-Personalize (SI Iter=1)           12.85 ± 3.18  26.20 ± 8.31   
LLM-Personalize (SI Iter=2)           27.25 ± 4.23  35.40 ± 7.31   
LITE-Planner                          26.97 ± 3.19  41.80 ± 3.08   

scene name                                 Scene 2                \
task set                                     train          test   
variant name                                                       
LLM-Personalize (Imitation learning)   5.13 ± 3.06  16.80 ± 5.81   
LLM-Personalize (SI Iter=1)           13.98 ± 3.04  17.20 ± 5.34   
LLM-Personalize (SI Iter=2)           21.13 ± 3.26  20.67 ± 7.15   
LITE-Planner                          22.88 ± 2.30  20.80 ± 4.36   

scene name                                 Scene 3                \
task set                                     train          test   
variant name                                                       
LLM-Personalize (Imitation learning)  28.55 ± 2.76  35.13 ± 4.60   
LLM-Personalize (SI Iter=1)           30.20 ± 3.28  30.00 ± 5.90   
LLM-Personalize (SI Iter=2)           32.57 ± 3.68  36.93 ± 4.98   
LITE-Planner                          41.22 ± 2.63  46.13 ± 4.51   

scene name                                 Scene 4                
task set                                     train          test  
variant name                                                      
LLM-Personalize (Imitation learning)  28.67 ± 2.79  29.60 ± 5.72  
LLM-Personalize (SI Iter=1)           28.45 ± 3.40  39.13 ± 7.63  
LLM-Personalize (SI Iter=2)           42.07 ± 2.88  34.13 ± 6.99  
LITE-Planner                          43.32 ± 3.40  37.20 ± 4.54



=== [표 2] 확장 씬 성공률 (x2, x3, x4) ===


scene name                     Scene 1 x2              
task set                            train          test
variant name                                           
LLM-Personalize (SI Iter=2)  40.21 ± 2.40  41.50 ± 3.12
LITE-Planner                 25.90 ± 1.96  41.80 ± 2.79

In [12]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 1000)

grouped = df_main.groupby(['scene name', 'task set', 'variant name']).agg({
    'llm_calls': 'mean',
    'ppr_calls': 'mean',
    'total_steps': 'mean'
})

grouped['LLM Calls'] = grouped['llm_calls'].apply(lambda x: f"{x:.1f}" if x > 0 else '-')

ppr_formatted = []
for (scene, task, variant), val in grouped['ppr_calls'].items():
    if "LITE-Planner" in variant:  
        ppr_formatted.append(f"{val:.1f}")
    else:
        ppr_formatted.append("-")
grouped['PPR Calls'] = ppr_formatted

grouped['Avg Steps'] = grouped['total_steps'].apply(lambda x: f"{x:.1f}" if x > 0 else "-")

final_table = grouped[['LLM Calls', 'PPR Calls', 'Avg Steps']].stack().unstack(level=2)

final_table = final_table.sort_index(axis=0, level=[0, 1], ascending=[True, False])

if 'desired_order' in locals():
    existing_cols = [c for c in desired_order if c in final_table.columns]
    final_table = final_table[existing_cols]

final_table

variant name                  LLM-Personalize (Imitation learning) LLM-Personalize (SI Iter=1) LLM-Personalize (SI Iter=2) LITE-Planner
scene name task set                                                                                                                    
Scene 1    train    LLM Calls                                  5.0                        13.0                        15.9          4.7
                    PPR Calls                                    -                           -                           -          0.0
                    Avg Steps                                588.8                       898.3                       916.7        552.8
           test     LLM Calls                                  5.1                        18.8                        15.4          4.8
                    PPR Calls                                    -                           -                           -          0.0
                    Avg Steps                                603.9                       934.7                       960.5        556.6
Scene 1 x2 train    LLM Calls                                    -                           -                        12.5          7.8
                    PPR Calls                                    -                           -                           -          0.0
                    Avg Steps                                    -                           -                       952.0        771.6
           test     LLM Calls                                    -                           -                        12.8          7.7
                    PPR Calls                                    -                           -                           -          0.0
                    Avg Steps                                    -                           -                       959.1        796.2
Scene 1 x3 train    LLM Calls                                    -                           -                           -            -
                    PPR Calls                                    -                           -                           -          nan
                    Avg Steps                                    -                           -                           -            -
           test     LLM Calls                                    -                           -                           -            -
                    PPR Calls                                    -                           -                           -          nan
                    Avg Steps                                    -                           -                           -            -
Scene 1 x4 train    LLM Calls                                    -                           -                           -            -
                    PPR Calls                                    -                           -                           -          nan
                    Avg Steps                                    -                           -                           -            -
           test     LLM Calls                                    -                           -                           -            -
                    PPR Calls                                    -                           -                           -          nan
                    Avg Steps                                    -                           -                           -            -
Scene 2    train    LLM Calls                                  7.1                        13.6                        20.3          5.5
                    PPR Calls                                    -                           -                           -          0.0
                    Avg Steps                                802.8                       638.8                       771.3        578.9
           test     LLM Calls                           

In [13]:
target_variant = "LLM-Personalize (Imitation learning)"
target_scene = "Scene 1"
target_split = "train"  # 또는 "test"

# 'split' 컬럼(또는 'task set') 조건을 추가합니다.
result = df_main[
    (df_main['variant name'] == target_variant) & 
    (df_main['scene name'] == target_scene) &
    (df_main['split'] == target_split)  # 데이터프레임의 실제 컬럼명 확인 필요
]['total_steps'].describe()

print(result)

count    100.000000
mean     588.820000
std      270.688922
min        0.000000
25%      407.000000
50%      576.000000
75%      854.750000
max      993.000000
Name: total_steps, dtype: float64


In [14]:
import glob
import os

# 1. 코드가 검색하고 있는 경로
test_exp_id = "ours_scene1_pair_eval_x2"
test_pattern = f"{LOGS_FOLDER}/{test_exp_id}/demo/pomaria_1_int/data_*.json"

# 2. 파일 검색 결과 확인
files = glob.glob(test_pattern)
print(f"🔍 탐색 경로: {test_pattern}")
print(f"📄 찾은 파일 개수: {len(files)}개")

# 3. 파일이 0개라면, 상위 폴더라도 존재하는지 확인
exp_folder_path = f"{LOGS_FOLDER}/{test_exp_id}"
print(f"📁 실험 폴더 존재 여부: {os.path.exists(exp_folder_path)}")

🔍 탐색 경로: /workspace/codellmpersonalize/logs/ours_scene1_pair_eval_x2/demo/pomaria_1_int/data_*.json
📄 찾은 파일 개수: 125개
📁 실험 폴더 존재 여부: True
